# 09 — LangGraph NBA Agent

## Objective
Wrap the deterministic Next Best Action engine behind a LangGraph workflow.

The important architectural rule is:

> **The LLM / conversational layer does not invent product, channel, propensity, or expected value.**

Those values come from the frozen CatBoost + NBA engine.

This notebook demonstrates:
1. loading a customer;
2. calling the deterministic NBA tool;
3. retrieving product details;
4. requiring confirmation before execution;
5. simulating offer delivery;
6. human handoff when confirmation is not granted.

The graph runs without any external API key so that the hackathon demo is reproducible.
Later, an LLM node can be inserted for natural-language conversation while preserving the same deterministic tools.


## 1. Install dependencies

In [13]:
!pip -q install langgraph langchain catboost pyarrow scikit-learn joblib

## 2. Mount Drive and imports

In [14]:

from pathlib import Path
import sys, json, joblib
import pandas as pd
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

PROJECT_DIR = Path('/content/drive/MyDrive/03 Projects/Factored Hackathon')
SRC_DIR = PROJECT_DIR / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from propensity.config import ARTIFACT_DIR, NUMERIC_FEATURES, CATEGORICAL_FEATURES
from propensity.features import INTERACTION_FEATURES
from propensity.langgraph_nba_agent import AgentContext, build_nba_graph

FINAL_MODEL_DIR = Path(ARTIFACT_DIR) / 'final_evaluation' / 'catboost_20261004_222052'
NBA_DIR = Path(ARTIFACT_DIR) / 'nba'

MODEL_PATH = FINAL_MODEL_DIR / 'model.joblib'
ACTION_CATALOG_PATH = NBA_DIR / 'action_catalog_pretest.parquet'
CUSTOMER_SNAPSHOT_PATH = NBA_DIR / 'customer_snapshot_pretest.parquet'

for p in [MODEL_PATH, ACTION_CATALOG_PATH, CUSTOMER_SNAPSHOT_PATH]:
    assert p.exists(), f'Missing required artifact: {p}'

print('All required artifacts found.')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
All required artifacts found.


## 3. Load model + NBA artifacts

In [15]:

model = joblib.load(MODEL_PATH)
action_catalog = pd.read_parquet(ACTION_CATALOG_PATH)
customer_snapshot = pd.read_parquet(CUSTOMER_SNAPSHOT_PATH)

customer_id_col = next(
    c for c in ['customer_id', 'client_id', 'user_id']
    if c in customer_snapshot.columns
)

MODEL_FEATURES_NBA = NUMERIC_FEATURES + CATEGORICAL_FEATURES + INTERACTION_FEATURES
CATEGORICAL_FEATURES_NBA = CATEGORICAL_FEATURES + INTERACTION_FEATURES

ctx = AgentContext(
    model=model,
    action_catalog=action_catalog,
    customer_snapshot=customer_snapshot,
    model_features=MODEL_FEATURES_NBA,
    categorical_features=CATEGORICAL_FEATURES_NBA,
    customer_id_col=customer_id_col,
)

print('Customers:', f'{len(customer_snapshot):,}')
print('Action profiles:', len(action_catalog))
print('Customer ID column:', customer_id_col)


Customers: 149,835
Action profiles: 60
Customer ID column: customer_id


## 4. Build LangGraph

In [16]:
agent = build_nba_graph(ctx)

graph = agent.get_graph()

print("LangGraph built successfully.")
print("\nNodes:")
for node_name in graph.nodes:
    print("-", node_name)

print("\nEdges:")
for edge in graph.edges:
    print(f"- {edge.source} -> {edge.target}")


LangGraph built successfully.

Nodes:
- __start__
- load_customer
- recommend
- prepare_offer
- no_action
- execute_offer
- handoff
- __end__

Edges:
- __start__ -> load_customer
- load_customer -> recommend
- prepare_offer -> execute_offer
- prepare_offer -> handoff
- recommend -> no_action
- recommend -> prepare_offer
- execute_offer -> __end__
- handoff -> __end__
- no_action -> __end__


## 5. Pick a deterministic demo customer

In [17]:

eligible = customer_snapshot.copy()
if 'accepts_marketing' in eligible.columns:
    eligible = eligible[
        ~eligible['accepts_marketing'].astype(str).str.lower().isin(['false', '0', 'no', 'n'])
    ]

DEMO_CUSTOMER_ID = str(eligible.iloc[0][customer_id_col])
print('Demo customer:', DEMO_CUSTOMER_ID)


Demo customer: CLI-P21780PQ8D9W


## 6. Demo A — customer confirms

In [18]:

result_confirmed = agent.invoke({
    'customer_id': DEMO_CUSTOMER_ID,
    'user_message': 'Show me the best offer and proceed if I confirm.',
    'user_confirmed': True,
})

print('STATUS:', result_confirmed['status'])
print('ASSISTANT:', result_confirmed.get('assistant_message'))
print('\nRECOMMENDATION:')
print(pd.Series(result_confirmed.get('recommendation', {})))

print('\nEXECUTION RESULT:')
print(pd.Series(result_confirmed.get('execution_result', {})))

if result_confirmed.get('ranked_actions'):
    print('\nTOP 5 CANDIDATE ACTIONS:')
    display(pd.DataFrame(result_confirmed['ranked_actions']).head(5))


STATUS: COMPLETED
ASSISTANT: Offer simulated successfully for Tarjeta Crédito via Push.

RECOMMENDATION:
customer_id                  CLI-P21780PQ8D9W
decision                               ACTION
promoted_product              Tarjeta Crédito
send_channel                             Push
propensity_raw                       0.008406
expected_conversion_value             3186.94
estimated_send_cost                    0.0005
expected_value                      26.788733
historical_support                       5887
dtype: object

EXECUTION RESULT:
status                             SIMULATED_SENT
customer_id                      CLI-P21780PQ8D9W
product                           Tarjeta Crédito
channel                                      Push
provider_message_id    demo-CLI-P21780PQ8D9W-Push
dtype: object

TOP 5 CANDIDATE ACTIONS:


,country,promoted_product,send_channel,campaign_type,campaign_objective,target_segment,target_country,expected_conversion_rate,send_hour,send_day_of_week,send_month,is_weekend,estimated_send_cost,estimated_conversion_value,historical_sends,historical_conversions,historical_conversion_rate,propensity_raw,expected_value,rank
0,Argentina,Tarjeta Crédito,Push,Mix,Acquisition,Plus,Mexico,9.35,12.0,3.0,6.0,0.0,0.0005,3186.940,5887,49,0.008323,0.008406,26.788733,1
1,Argentina,Préstamo Hipotecario,SMS,SMS,Cross-sell,Student,Mexico,6.53,11.0,3.0,3.0,0.0,0.0923,2851.845,2325,12,0.005161,0.009133,25.953558,2
2,Argentina,Cuenta Corriente,SMS,SMS,Cross-sell,Student,Colombia,11.75,12.0,3.0,9.0,0.0,0.0915,2673.305,8860,70,0.007901,0.009538,25.406779,3
3,Argentina,Inversión,Push,Push,Retention,Student,Mexico,10.76,12.0,3.0,6.0,0.0,0.0005,2785.300,11032,81,0.007342,0.008484,23.630380,4
4,Argentina,Tarjeta Crédito,SMS,SMS,Retention,Premium,Colombia,10.29,11.0,3.0,8.0,0.0,0.0911,2415.525,27590,284,0.010294,0.009169,22.056919,5


## 7. Demo B — customer does NOT confirm

In [19]:

result_not_confirmed = agent.invoke({
    'customer_id': DEMO_CUSTOMER_ID,
    'user_message': 'Show me the recommendation but do not send anything.',
    'user_confirmed': False,
})

print('STATUS:', result_not_confirmed['status'])
print('ASSISTANT:', result_not_confirmed.get('assistant_message'))
print('\nHANDOFF RESULT:')
print(pd.Series(result_not_confirmed.get('execution_result', {})))


STATUS: HANDOFF
ASSISTANT: No automated send performed. A human handoff was created.

HANDOFF RESULT:
status                                       HANDOFF_CREATED
customer_id                                 CLI-P21780PQ8D9W
reason         Customer did not confirm automated execution.
queue                                       sales-assistance
dtype: object


## 8. Inspect the deterministic recommendation payload

In [20]:

rec = result_confirmed.get('recommendation', {})

important_fields = {
    'customer_id': rec.get('customer_id'),
    'decision': rec.get('decision'),
    'product': rec.get('promoted_product'),
    'channel': rec.get('send_channel'),
    'propensity_raw': rec.get('propensity_raw'),
    'expected_conversion_value': rec.get('expected_conversion_value'),
    'estimated_send_cost': rec.get('estimated_send_cost'),
    'expected_value': rec.get('expected_value'),
    'historical_support': rec.get('historical_support'),
}
display(pd.Series(important_fields))

assert rec.get('decision') in {
    'ACTION',
    'NO_ACTION_CONSENT',
    'NO_ACTION_NEGATIVE_VALUE',
    'NO_ACTION_NO_SUPPORTED_CANDIDATES',
}


,0
customer_id,CLI-P21780PQ8D9W
decision,ACTION
product,Tarjeta Crédito
channel,Push
propensity_raw,0.008406
expected_conversion_value,3186.94
estimated_send_cost,0.0005
expected_value,26.788733
historical_support,5887


## 9. Persist demo results

In [21]:

LANGGRAPH_DIR = NBA_DIR / 'langgraph_demo'
LANGGRAPH_DIR.mkdir(parents=True, exist_ok=True)

summary = {
    'demo_customer_id': DEMO_CUSTOMER_ID,
    'confirmed_status': result_confirmed.get('status'),
    'confirmed_recommendation': result_confirmed.get('recommendation'),
    'confirmed_execution': result_confirmed.get('execution_result'),
    'not_confirmed_status': result_not_confirmed.get('status'),
    'not_confirmed_execution': result_not_confirmed.get('execution_result'),
    'architecture_note': (
        'LangGraph orchestrates deterministic NBA tools. '
        'The conversational layer does not invent product, channel, propensity, or expected value.'
    ),
}

summary_path = LANGGRAPH_DIR / 'demo_summary.json'
summary_path.write_text(json.dumps(summary, indent=2, default=str), encoding='utf-8')

print('Saved:', summary_path)


Saved: /content/drive/MyDrive/03 Projects/Factored Hackathon/artifacts/propensity/nba/langgraph_demo/demo_summary.json


## What this proves

At this point the end-to-end decision workflow is:

```text
Customer
  ↓
LangGraph
  ↓
get_next_best_action(customer_id)
  ↓
Frozen CatBoost propensity model
  ↓
Economic ranking
  ↓
Product + channel + probability + expected value
  ↓
Confirmation gate
  ├─ Yes → simulated send
  └─ No  → human handoff
```

### Next step
Move this graph into the deployable application layer (FastAPI / UI) and optionally insert an LLM node for natural-language explanation and Q&A.

The deterministic NBA function remains the source of truth.
